## **PAIR ASSIGNMENT**

Please work in pairs for this exercise. Pair up with the person next to you. If you find that there isn't anyone sitting next to you or if you're unable to form a pair, please raise your hand, and I will assist in pairing you with someone.


### **Task 0: Please assign your full name and your partner's full name to the variables below, respectively.**


Example:

```
your_name = 'Taylor Swift'
your_partner_name = 'Travis Kelce'
```

In [ ]:
# Assign your and your partner's names here.  If you find that there isn't anyone sitting next to you or if you're unable to form a pair, please raise your hand.
# If, in the end, we are unable to find a partner for you, please assign the word "self" to the `your_partner_name` variable.
your_name = 'Jonathan Cruz'
your_partner_name = ''

# **SQL for Data Science**

In this notebook exercise, we will explore **SQL (Structured Query Language)** and **SQLite**, learning how to create databases, store data, and query it effectively. These are fundamental skills for any data scientist working with structured data.

## **What is SQL?**

**SQL (Structured Query Language)** is a standard programming language designed for managing and manipulating relational databases. It allows you to:

- **Create** databases and tables to store structured data
- **Insert** new data into tables
- **Query** data using SELECT statements to retrieve specific information
- **Update** existing records
- **Delete** data when needed
- **Aggregate** data using functions like COUNT, SUM, AVG, etc.

SQL is the lingua franca of data storage and retrieval. Whether you're working with MySQL, PostgreSQL, Oracle, SQL Server, or SQLite, the core SQL syntax remains largely the same.

## **What is SQLite?**

**SQLite** is a lightweight, serverless, self-contained SQL database engine. Unlike other database systems (MySQL, PostgreSQL), SQLite:

- **Requires no server**: The entire database is stored in a single file on disk
- **Zero configuration**: No setup or administration needed
- **Portable**: Database files can be copied and shared across different platforms
- **Built into Python**: The `sqlite3` module is part of Python's standard library
- **Great for prototyping**: Perfect for learning SQL and building small to medium applications

SQLite is used in many applications including web browsers (Chrome, Firefox), mobile apps (iOS, Android), and as embedded databases in countless software products.

## **Objectives**

In this exercise, you will:
1. Create a SQLite database and a table
2. Load data from scikit-learn's 20 Newsgroups dataset into the database
3. Write SQL queries to explore and analyze the data
4. Learn common SQL operations like COUNT, DISTINCT, and LIKE

## **Import Libraries**

Let's start by importing the necessary libraries for our analysis.

In [1]:
import sqlite3
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

## **Load the 20 Newsgroups Dataset**

The 20 Newsgroups dataset is a collection of approximately 20,000 newsgroup documents, partitioned across 20 different newsgroups. It's commonly used for text classification and natural language processing tasks.

- Load the dataset using `fetch_20newsgroups()`
- Explore the structure of the data
- Print some sample documents and their group names

In [3]:
# Write your code here
print("Loading 20 Newsgroups dataset...")
newsgroups = fetch_20newsgroups(data_home=".", subset='all', remove=('headers', 'footers', 'quotes'))

print(f"\nNumber of documents: {len(newsgroups.data)}")
print(f"Number of groups: {len(newsgroups.target_names)}")
print(f"\nGroup names:")
for i, name in enumerate(newsgroups.target_names):
    print(f"  {i}: {name}")

Loading 20 Newsgroups dataset...

Number of documents: 18846
Number of groups: 20

Group names:
  0: alt.atheism
  1: comp.graphics
  2: comp.os.ms-windows.misc
  3: comp.sys.ibm.pc.hardware
  4: comp.sys.mac.hardware
  5: comp.windows.x
  6: misc.forsale
  7: rec.autos
  8: rec.motorcycles
  9: rec.sport.baseball
  10: rec.sport.hockey
  11: sci.crypt
  12: sci.electronics
  13: sci.med
  14: sci.space
  15: soc.religion.christian
  16: talk.politics.guns
  17: talk.politics.mideast
  18: talk.politics.misc
  19: talk.religion.misc


In [4]:
# Let's look at a sample document
print("Sample document:")
print("=" * 50)
print(f"Group: {newsgroups.target_names[newsgroups.target[0]]}")
print(f"Text (first 500 chars):\n{newsgroups.data[0][:500]}")

Sample document:
Group: rec.sport.hockey
Text (first 500 chars):


I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a


## **TASK 1: Create a SQLite Database and Table**

Now let's create a SQLite database and a table to store our newsgroups data.

- Create a connection to a new SQLite database called `newsgroups.db`
- Create a table named `newsgroups` with two columns:
  - `RawText` (TEXT): The content of the newsgroup post
  - `GroupName` (TEXT): The name of the newsgroup
- Use `IF NOT EXISTS` to avoid errors if the table already exists

**SQL Syntax for CREATE TABLE:**
```sql
CREATE TABLE IF NOT EXISTS table_name (
    column1 TYPE,
    column2 TYPE
);
```

In [5]:
# Write your code here

# Create a connection to the database (this creates the file if it doesn't exist)
conn = sqlite3.connect("newsgroups.db")

# Create a cursor object to execute SQL commands
cursor = conn.cursor()

# Drop the table if it exists (to start fresh)
cursor.execute('DROP TABLE IF EXISTS newsgroups')

# Create the newsgroups table with RawText and GroupName columns
cursor.execute("""
CREATE TABLE IF NOT EXISTS newsgroups (
    RawText TEXT,
    GroupName TEXT
)
""")

# Commit the changes
conn.commit()

print("Database 'newsgroups.db' created successfully!")
print("Table 'newsgroups' created with columns: RawText, GroupName")

Database 'newsgroups.db' created successfully!
Table 'newsgroups' created with columns: RawText, GroupName


## **TASK 2: Load Data into the Database**

Now let's insert the newsgroups data into our SQLite table.

- Loop through the newsgroups data
- For each document, insert the raw text and corresponding group name
- Use parameterized queries (with `?` placeholders) to avoid SQL injection
- Commit the changes after inserting all data

**SQL Syntax for INSERT:**
```sql
INSERT INTO table_name (column1, column2) VALUES (?, ?)
```

**Tip:** Use `executemany()` for efficient bulk insertion.

In [6]:
# Prepare the data as a list of tuples (RawText, GroupName)
data_to_insert = []
for i in range(len(newsgroups.data)):
    raw_text = newsgroups.data[i]
    group_name = newsgroups.target_names[newsgroups.target[i]]
    data_to_insert.append((raw_text, group_name))

print(f"Prepared {len(data_to_insert)} records for insertion...")

# Write your code here

# Insert all data using executemany for efficiency
cursor.executemany("INSERT INTO newsgroups (RawText, GroupName) VALUES (?, ?)", data_to_insert)

# Commit the changes
conn.commit()

print(f"Successfully inserted {len(data_to_insert)} records into the database!")

Prepared 18846 records for insertion...
Successfully inserted 18846 records into the database!


## **TASK 3: Query the Database - Count Total Rows**

Let's verify our data by counting the total number of rows in the table.

- Use the `COUNT(*)` function to count all rows
- Execute the query and fetch the result

**SQL Syntax:**
```sql
SELECT COUNT(*) FROM table_name
```

In [7]:
# Write your code here

cursor.execute("SELECT COUNT(*) FROM newsgroups")
total_rows = cursor.fetchone()[0]

print(f"Total number of rows in the newsgroups table: {total_rows}")

Total number of rows in the newsgroups table: 18846


## **TASK 4: Query with LIKE - Find Posts Containing 'fun'**

Let's find out how many posts contain the word 'fun' in the RawText column.

- Use the `LIKE` operator with wildcards (`%`) for pattern matching
- The pattern `%fun%` matches any text containing 'fun' anywhere

**SQL Syntax:**
```sql
SELECT COUNT(*) FROM table_name WHERE column LIKE '%pattern%'
```

**Note:** SQLite's LIKE is case-insensitive by default for ASCII characters.

In [8]:
# Write your code here
cursor.execute("""
SELECT COUNT(*) 
FROM newsgroups 
WHERE RawText LIKE '%fun%'
""")
fun_count = cursor.fetchone()[0]

print(f"Number of posts containing 'fun': {fun_count}")
print(f"Percentage of total posts: {fun_count / total_rows * 100:.2f}%")

Number of posts containing 'fun': 1252
Percentage of total posts: 6.64%


In [9]:
# Let's look at a sample post containing 'fun'
cursor.execute("""
SELECT RawText, GroupName
FROM newsgroups
WHERE RawText LIKE '%fun%'
LIMIT 1
""")

sample = cursor.fetchone()
print("Sample post containing 'fun':")
print("=" * 50)
print(f"Group: {sample[1]}")
print(f"Text (first 500 chars):\n{sample[0][:500]}")

Sample post containing 'fun':
Group: rec.sport.hockey
Text (first 500 chars):


I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a


## **TASK 5: Query with DISTINCT - Count Unique Group Names**

Let's count how many distinct (unique) group names are in our database.

- Use `COUNT(DISTINCT column)` to count unique values
- Also retrieve the list of all distinct group names

**SQL Syntax:**
```sql
SELECT COUNT(DISTINCT column) FROM table_name
SELECT DISTINCT column FROM table_name
```

In [10]:
# Write your code here

cursor.execute("SELECT COUNT(DISTINCT GroupName) FROM newsgroups")
distinct_groups = cursor.fetchone()[0]

print(f"Number of distinct group names: {distinct_groups}")

Number of distinct group names: 20


In [11]:
# Query to get all distinct group names
cursor.execute("SELECT DISTINCT GroupName FROM newsgroups")
group_names = cursor.fetchall()

print("\nAll distinct group names:")
for i, (name,) in enumerate(group_names, 1):
    print(f"  {i}. {name}")


All distinct group names:
  1. rec.sport.hockey
  2. comp.sys.ibm.pc.hardware
  3. talk.politics.mideast
  4. comp.sys.mac.hardware
  5. sci.electronics
  6. talk.religion.misc
  7. sci.crypt
  8. sci.med
  9. alt.atheism
  10. rec.motorcycles
  11. rec.autos
  12. comp.windows.x
  13. comp.graphics
  14. sci.space
  15. talk.politics.guns
  16. misc.forsale
  17. rec.sport.baseball
  18. talk.politics.misc
  19. comp.os.ms-windows.misc
  20. soc.religion.christian


## **BONUS: More SQL Queries**

Let's explore a few more useful SQL queries:

1. Count posts per group using `GROUP BY`
2. Find groups with posts containing specific keywords
3. Use `ORDER BY` to sort results

In [ ]:
# Count posts per group, sorted by count (descending)
cursor.execute(...)

print("Posts per group:")
print("=" * 50)
for group_name, count in cursor.fetchall():
    print(f"{group_name}: {count} posts")

In [ ]:
# Find how many posts in each group contain 'computer'
cursor.execute(...)

print("\nTop 10 groups with posts containing 'computer':")
print("=" * 50)
for group_name, count in cursor.fetchall():
    print(f"{group_name}: {count} posts")

In [ ]:
# Using pandas to display results in a nice table format
query = ...

df = pd.read_sql_query(query, conn)
df['PercentWithFun'] = (df['PostsWithFun'] / df['TotalPosts'] * 100).round(2)
print("\nAnalysis of posts containing 'fun' by group:")
df

## **Clean Up**

Always remember to close the database connection when you're done.

In [12]:
# Close the database connection
conn.close()
print("Database connection closed.")

Database connection closed.


## **Summary and Key Takeaways**

### What we learned:

1. **SQL Basics**:
   - SQL is the standard language for working with relational databases
   - Core operations include CREATE, INSERT, SELECT, UPDATE, and DELETE
   - SQL is used across many different database systems

2. **SQLite**:
   - SQLite is a lightweight, file-based database
   - Requires no server setup or configuration
   - Built into Python via the `sqlite3` module
   - Great for learning, prototyping, and small applications

3. **Key SQL Operations**:
   - `CREATE TABLE`: Define the structure of your data
   - `INSERT INTO`: Add data to your tables
   - `SELECT`: Retrieve data from tables
   - `COUNT(*)`: Count the number of rows
   - `COUNT(DISTINCT column)`: Count unique values
   - `LIKE '%pattern%'`: Pattern matching for text search
   - `GROUP BY`: Aggregate data by categories
   - `ORDER BY`: Sort your results

4. **Best Practices**:
   - Use parameterized queries (`?` placeholders) to prevent SQL injection
   - Use `executemany()` for bulk insertions
   - Always close your database connections when done
   - Use pandas' `read_sql_query()` for convenient data analysis

### Discussion Questions:

1. When would you choose SQLite over a full database server like PostgreSQL?
2. How does SQL compare to pandas for data manipulation?
3. What are the advantages of storing data in a database versus a CSV file?